In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ (her notebook'un ilk hücresi) ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "a/1.3-missing"                                  # merge edildikten sonra "main"
try:
    TOKEN = userdata.get("GH_TOKEN")                        # sadece private repo için gerekir
except userdata.SecretNotFoundError:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .
if "/content/repo/src" not in sys.path:      # editable install is only seen after a kernel restart
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()          # config/config.yaml + DATA_ROOT
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 03 — Missing values

Stage 1.3. This notebook contains no business logic; it only calls `recsys.cleaning.missing`.

* Inputs: stages **`loaded`** and **`quality`** (LATEST), validated against their schemas.
* Output: one **`missing`** version with the evidence tables below (each validated against
  `MISSING_OUTPUT_SCHEMA`).
* The analysis uses the full data because it is exploratory. **The transformers are only
  demonstrated here: their real fit happens in the 1.9 pipeline, on train data only.**

In [ ]:
from recsys.cleaning.missing import (
    DEFAULT_MAIN_TAGS,
    DEFAULT_NO_COOK_TAGS,
    GroupMedianImputer,
    RatingZeroToNaN,
    ZeroAsMissing,
    main_tag_coverage,
    mark_has_rating,
    minutes_zero_examples,
    minutes_zero_n_steps,
    minutes_zero_summary,
    minutes_zero_tag_lift,
    missing_rates,
    rating_zero_examples,
    rating_zero_review_stats,
)
from recsys.contracts import (
    LOADED_INTERACTIONS_SCHEMA,
    LOADED_RECIPES_SCHEMA,
    MISSING_OUTPUT_SCHEMA,
    QUALITY_ISSUES_SCHEMA,
    validate,
)
from recsys.io import ArtifactStore

N_EXAMPLES = 10
TOP_TAGS = 20
NO_COOK_TAGS = DEFAULT_NO_COOK_TAGS   # tags tested as evidence for a genuine zero (section 3)
RELATED_ISSUES = (
    "rating_zero",
    "minutes_zero",
    "missing_name",
    "missing_description",
    "missing_review",
    "recipes_without_rating",
)

store = ArtifactStore.from_config(CFG)
loaded = store.load("loaded")
quality = store.load("quality")
recipes = validate(loaded["recipes"], LOADED_RECIPES_SCHEMA)
interactions = validate(loaded["interactions"], LOADED_INTERACTIONS_SCHEMA)
issues = validate(quality["issues"], QUALITY_ISSUES_SCHEMA)
print("inputs: loaded", loaded.version, "| quality", quality.version)
print("recipes", recipes.shape, "| interactions", interactions.shape)

## 1. Missing values

`kind = null` is a real NaN, `empty_list` an empty `[]`, and `zero` a 0 that may encode "missing"
(`rating = 0` and `minutes = 0`). The second table is the part of the 1.2 issues table this
stage is responsible for.

In [ ]:
rates = missing_rates(recipes, interactions)
related = issues[issues["issue_id"].isin(RELATED_ISSUES)].reset_index(drop=True)
display(related)
rates[rates["n_missing"] > 0]

## 2. rating = 0

On Food.com a 0 means that no stars were given. If these rows still carry review text, the 0 is
not a broken record but an unrated review, so it becomes NaN (`RatingZeroToNaN`) instead of
being treated as the worst score.

In [ ]:
zero_stats = rating_zero_review_stats(interactions)
zero_examples = rating_zero_examples(interactions, n=N_EXAMPLES, random_state=CFG.random_state)
display(zero_stats.T)
zero_examples

## 3. minutes = 0: genuine no-cook recipe or missing value?

`minutes` is the total preparation time. Evidence for "genuine zero": the recipe is tagged as
no-cook / beverage, has few steps and few ingredients. Evidence for "missing": a 0 on recipes
that clearly need cooking (many steps, cooking tags such as `oven`).

**Decision:** zero-minute recipes are not over-represented among no-cook / beverage recipes,
have as many steps as the catalogue and over-represent `oven`, so `minutes = 0` is treated as a
missing value for every recipe (no tag exception) and imputed by `GroupMedianImputer`.
`15-minutes-or-less` appears on all of them because Food.com derives that tag from `minutes`,
so it is not evidence.

In [ ]:
mz_summary = minutes_zero_summary(recipes, NO_COOK_TAGS)
mz_tags = minutes_zero_tag_lift(recipes, top_n=TOP_TAGS)
display(mz_summary.T)
mz_tags

In [ ]:
mz_steps = minutes_zero_n_steps(recipes)
mz_examples = minutes_zero_examples(recipes, n=N_EXAMPLES, random_state=CFG.random_state)
display(mz_steps.head(15))
mz_examples

## 4. Recipes without a rating

`has_rating = False` recipes have no rating > 0. They stay in the recommendation pool but are not
used for training.

In [ ]:
flagged = mark_has_rating(recipes, interactions)
flagged["has_rating"].value_counts()

## 5. Transformer demo (not the real fit)

The real fit happens in 1.9 on train data only. Here the imputer is fitted on a random share of
the recipes just to show the learned groups and how often the global fallback would be used.

In [ ]:
after_rating = RatingZeroToNaN().fit_transform(interactions)
print("rating NaN after RatingZeroToNaN:", int(after_rating["rating"].isna().sum()))

# Decision (section 3 evidence): every minutes = 0 is a missing-value code, no tag exception.
zero_missing = ZeroAsMissing(columns=("minutes",))
recipes_z = zero_missing.fit_transform(recipes)
print("minutes NaN after ZeroAsMissing:", int(recipes_z["minutes"].isna().sum()))

coverage = main_tag_coverage(recipes, DEFAULT_MAIN_TAGS)
demo_fit = recipes_z.sample(frac=CFG.split["train"], random_state=CFG.random_state)
imputer = GroupMedianImputer().fit(demo_fit)
print("global median (demo fit):", imputer.global_median_)
print("fallback share on the other rows:", imputer.fallback_share(recipes_z.drop(demo_fit.index)))
display(coverage)
imputer.group_table_

## 6. Save stage `missing`

In [ ]:
missing = store.save(
    "missing",
    {
        "missing_rates": rates,
        "related_issues": related,
        "rating_zero_review_stats": zero_stats,
        "rating_zero_examples": zero_examples,
        "minutes_zero_summary": mz_summary,
        "minutes_zero_tags": mz_tags,
        "minutes_zero_n_steps": mz_steps,
        "minutes_zero_examples": mz_examples,
        "main_tag_coverage": coverage,
        "imputer_groups": imputer.group_table_,
    },
    inputs=[loaded, quality],
    params={
        "n_examples": N_EXAMPLES,
        "top_tags": TOP_TAGS,
        "no_cook_tags": NO_COOK_TAGS,
        "main_tags": DEFAULT_MAIN_TAGS,
        "imputer_params": imputer.get_params(),
        "demo_fit_frac": CFG.split["train"],
        "random_state": CFG.random_state,
    },
    schemas=MISSING_OUTPUT_SCHEMA,
)
print("saved:", missing.stage, missing.version, "->", missing.path)
check = store.load("missing")  # re-read with sha256 verification
{name: f["n_rows"] for name, f in check.manifest["files"].items()}